# Tanım

1. Metnin duygusunu (pozitif/negatif/nötr) sınıflandırma

- Bu ürün harika, kesinlikle tavsiye ederim -> Pozitif
- Kalitesi berbat, boşa para -> negatif
- Ürün geldi, kullanıyorum -> nötr

# 3. Yaklaşım

1. Yaklaşım             Araç                             Doğruluk

- Lexicon-based.      VADER, SentiWordNet.             Basit, sınırlı
- ML Klasik           NB, LogReg, SVM + TF-IDF            % 85-90
- Deep Learning       BERT, DistilBert                    %92-95
- LLM(GPT)            prompt enginnering                  %90-95

### Zorluklar

- Olumsuzluk: "iyi değil" != iyi
- İroni/sarkazm: "Harika! Yine bozuldu" -> negatif
- Bağlam: "küçük" -> "iyi mi kötü mü (ürün cinsine bağlı)
- Emoji,slang: "yaşasın" vs "yaşasın"
- Türkçe: eklerle olumsuzluk ("yapmadım"), dikkat

# Meşhur Datasetler

- IMDB (İngilzice 50k film yorumu)
- SST-2 (standford Sentiment Treebank)
- Yelp Reviews (restoran)
- Amazon Reviews (ürün, 142M)
- Türkçe: Yıldız Technical sentiment, Hepsiburada Turkish Sentiment


# Özet

- Merin -> pozitif/negatif/nötr sınıflandırma
- E-ticaret, sosyal medya, finans
- Lexicon -> ML -> BERT/LLM
- Olumsuzluk, ironi zor -> BERT bunda iyi 

# CountVectorizer + Naive Bayes

In [1]:
# --- MEKANIK: COUNTVectorizer metni nasıl sayıya cevirir?
# Kucuk oyuncak cumlelerle sadece "vektorleştirme"yi gosteriyoruz (model değerlendirmesi değil)

from sklearn.feature_extraction.text import CountVectorizer

toy_reviews = [
    "this movie is fantastic",
    "terrible ovie waste of time",
    "brillant acting and amazing story"
]

# Her cumle, kelime sayilardan olusan bir vektore donusur (Bag-of-Words)
vec_demo = CountVectorizer()
X_demo = vec_demo.fit_transform(toy_reviews)

print("Sozluk (feature isimleri):")
print(vec_demo.get_feature_names_out())
print("\nDokuman-terim matrisi (her satır bir cümle, her sutun bir kelime)")
print(X_demo.toarray())

Sozluk (feature isimleri):
['acting' 'amazing' 'and' 'brillant' 'fantastic' 'is' 'movie' 'of' 'ovie'
 'story' 'terrible' 'this' 'time' 'waste']

Dokuman-terim matrisi (her satır bir cümle, her sutun bir kelime)
[[0 0 0 0 1 1 1 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 1 1 0 1 0 1 1]
 [1 1 1 1 0 0 0 0 0 1 0 0 0 0]]


In [3]:
# --- Gercek VERI: NLTK movie_revies korpusu (2000) yorum
# Asil degerlendirmeyi bu gercek veri uzerinden yapiyoruz.
import numpy as np
import nltk
nltk.download("movie_reviews", quiet= True) # film yorumları korpusu (200 etiketli yorum - duygu analiz ve etiket)
from nltk.corpus import movie_reviews


documents = [(' '.join(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

np.random.seed(42)
np.random.shuffle(documents)

texts = []
labels = []
for t,l in documents:
    texts.append(t)
    labels.append(1 if l == 'pos' else 0) # pozitif = 1, negatif -> 0

print(f"Toplam yorum: {len(texts)} (pos = {sum(labels)}, neg = {len(labels)- sum(labels)})")
print(f"Ornek yorum (ilk 200 karakter): {texts[0][:200]}...")

Toplam yorum: 2000 (pos = 1000, neg = 1000)
Ornek yorum (ilk 200 karakter): the verdict : spine - chilling drama from horror maestro stephen king , featuring an outstanding , oscar - winning performance from kathy bates . geez , french and saunders had a field day when they s...


In [5]:
# --- EGITIM, CountVectorizer + MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

# Train/test ayrimi: %70 eğitim, %30 test
X_train,X_test,y_train,y_test = train_test_split(
    texts, labels, test_size= 0.3, random_state= 42, stratify= labels
)

# Metni Bag-of-Words vektorlere cevir (unigram + bigram)
vec = CountVectorizer(stop_words= 'english', ngram_range= (1,2), max_features = 10_000)
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

# Naive Bayes modelini eğit
model = MultinomialNB().fit(X_train_vec, y_train)

print(f"Test Accuracy (movie_reviews, {len(y_test)} ornek): {model.score(X_test_vec,y_test):.2%}")

Test Accuracy (movie_reviews, 600 ornek): 80.17%


In [6]:
# --- DEGERLENDİRME: Confusion Matrix + Classification Report, (gerceek test seti)
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test_vec)

print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred))
print("\nClassification Report:")
print(classification_report(y_test,y_pred, target_names = ['neg','pos'] ))

Confusion Matrix:
[[250  50]
 [ 69 231]]

Classification Report:
              precision    recall  f1-score   support

         neg       0.78      0.83      0.81       300
         pos       0.82      0.77      0.80       300

    accuracy                           0.80       600
   macro avg       0.80      0.80      0.80       600
weighted avg       0.80      0.80      0.80       600



In [7]:
# --- YORUMLAMA: En pozitif ve en negatif kelimeler ---
# log P(kelime|pos) - log P(kelime|neg) > 0 => poitif sinifi isaret eder
feature_names = vec.get_feature_names_out()
log_prob = model.feature_log_prob_
delta = log_prob[1] - log_prob[0] # pos - neg

top_pos = np.argsort(delta)[::-1][:10]
top_neg = np.argsort(delta)[:10]

print("En POZITIF kelimeler:")
for i in top_pos:
    print(f"    {feature_names[i]:<20} {delta[i]:+.2f}")

print("\nEn NEGATIF kelimeler:")
for i in top_neg:
    print(f"    {feature_names[i]:<20} {delta[i]:+.2f}")

En POZITIF kelimeler:
    lebowski             +3.72
    whale                +3.52
    gattaca              +3.50
    ordell               +3.47
    shrek                +3.47
    argento              +3.47
    black cauldron       +3.38
    leila                +3.35
    sweetback            +3.32
    big lebowski         +3.29

En NEGATIF kelimeler:
    nbsp                 -4.19
    mission mars         -3.45
    brenner              -3.33
    farrellys            -3.29
    sinise               -3.29
    heckerling           -3.20
    liz                  -3.16
    hudson               -3.16
    mr magoo             -3.16
    hewitt               -3.16


In [10]:
# --- ORNEK TAHMINLER (egitilmis gercek model ile) ---
def tahmin_et(yorum):
    X_new = vec.transform([yorum])
    pred = model.predict(X_new)[0]
    prob = model.predict_proba(X_new)[0]
    label = "POS" if pred == 1 else "NEG"
    print(f"[{label}] guven = {prob[pred]:.2%} -> {yorum}")

tahmin_et("absolutely loved this filmi one of the best of the year")
tahmin_et("horrible and disappointing, a complete waste of time")
tahmin_et("the story was nice-out it got boring at times")

[POS] guven = 65.88% -> absolutely loved this filmi one of the best of the year
[NEG] guven = 99.73% -> horrible and disappointing, a complete waste of time
[NEG] guven = 85.89% -> the story was nice-out it got boring at times


# NB'nin Avantajları

- Çok hızlı eğitim (milyonlarda yorumda saniyeler)
- Az veri ile iyi çalışır
- Metin sınıflandırmada hala güçlü
- Feature log-probability ile yorumlanabilir

# Özet

- CountVectorizer + MultinomialNB pipeline
- movie_reviews'da - %80 accuracy
- En güçlü pozitif/negatif kelimeler yorumlanabilir
- Hızlı baseline


# TF-IDF 

In [1]:
# --- MEKANIK: TF-IDF metni nasıl agirliklandirir? ---
# Kucuk oyuncak cumlelerle sadece vektorlestirmeyi gosteriyoruz (model degerlendirmesi DEGIL).
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

toy_reviews = [
    "this movie is fantastic",
    "terrible movie waste of time",
    "brillant acting amazing story"
]

# TF-IDF: sik gecen ama az ayirt edici kelimeleri bastirir, nadir/ayirt edicileri one cikarir
vec_demo = TfidfVectorizer()
X_demo = vec_demo.fit_transform(toy_reviews)

# Her satir bir cumle, her sutun bir kelimeni TF-IDF agirligi
df_demo = pd.DataFrame(X_demo.toarray().round(2),
                       columns= vec_demo.get_feature_names_out())
print("TF-IDF agırlik matrisi:")
print(df_demo)

TF-IDF agırlik matrisi:
   acting  amazing  brillant  fantastic    is  movie    of  story  terrible  \
0     0.0      0.0       0.0       0.53  0.53   0.40  0.00    0.0      0.00   
1     0.0      0.0       0.0       0.00  0.00   0.36  0.47    0.0      0.47   
2     0.5      0.5       0.5       0.00  0.00   0.00  0.00    0.5      0.00   

   this  time  waste  
0  0.53  0.00   0.00  
1  0.00  0.47   0.47  
2  0.00  0.00   0.00  


In [3]:
# --- GERCEK VERI: NLTK movie_reviews korpusu (2000 yorum) ---
import numpy as np
import nltk
nltk.download('movie_revies', quiet = True) 
from nltk.corpus import movie_reviews

documents = [(' '.join(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

np.random.seed(42)
np.random.shuffle(documents)

texts = [t for t, _ in documents]
labels = []

for _ , c in documents:
    labels.append(1 if c == 'pos' else 0) # pozitif -> 1, negatif -> 0

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size= 0.3, random_state= 42, stratify= labels
)
print(f"Eğitim: {len(X_train)} | Test {len(X_test)} ornek")

[nltk_data] Error loading movie_revies: Package 'movie_revies' not
[nltk_data]     found in index


Eğitim: 1400 | Test 600 ornek


In [5]:
# --- BASELINE TF-IDF + LogisticRegression Pipeline ---
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('vec', TfidfVectorizer(
        stop_words= 'english',
        ngram_range= (1,2),
        max_features= 10_000,
        sublinear_tf= True
    )),
    ('clf', LogisticRegression(max_iter= 10000))
])

pipe.fit(X_train,y_train)

# HEADLINE: gercek veri uzerindeki durust baseline accuracy
print(f"Baseline Accuracy (movie_reviesw, {len(y_test)} test ornegi): {pipe.score(X_test, y_test):.2%}")

Baseline Accuracy (movie_reviesw, 600 test ornegi): 83.17%


In [7]:
# --- GRIDSEARCHCV: hiperparametre tuning (gercek veri uzerinde) ---
from sklearn.model_selection import GridSearchCV

# Hiz için makul bir parametre gridi; CV F1 ve Test F1 (~0.8))

param_grid = {
    'vec__ngram_range': [(1,1),(1,2)],
    'vec__max_features': [5000,10000],
    'clf__C': [0.1,1,10]
}

grid = GridSearchCV(pipe, param_grid, cv = 3, scoring= 'f1', n_jobs= -1)
grid.fit(X_train, y_train)

print(f"En iyi parametreler: {grid.best_params_}")
print(f"En iyi CV F1:        {grid.best_score_:.3f}")
print(f"Test F1:             {grid.score(X_test, y_test):.3f}")

En iyi parametreler: {'clf__C': 10, 'vec__max_features': 10000, 'vec__ngram_range': (1, 1)}
En iyi CV F1:        0.848
Test F1:             0.854


In [9]:
# --- YORUMLAMA: LogReg katsayilari (optimal model) ---
best_pipe = grid.best_estimator_
vec = best_pipe.named_steps['vec']
clf = best_pipe.named_steps['clf']

feature_names = vec.get_feature_names_out()
coefs = clf.coef_[0]

# Pozitif katsayi => pozitif sinifi, negatif katsayi => negatif sinifi isaret eder
top_pos_idx = np.argsort(coefs)[::-1][:10]
top_neg_idx = np.argsort(coefs)[:10]

print("En guclu POZITIF feature'lar:")
for i in top_pos_idx:
    print(f"    {feature_names[i]:<25} coef = {coefs[i]:+.3f}")


print("En guclu NEGATIF feature'lar:")
for i in top_neg_idx:
    print(f"    {feature_names[i]:<25} coef = {coefs[i]:+.3f}")

En guclu POZITIF feature'lar:
    hilarious                 coef = +3.420
    fun                       coef = +3.398
    world                     coef = +3.130
    terrific                  coef = +3.007
    great                     coef = +2.980
    overall                   coef = +2.811
    excellent                 coef = +2.757
    gives                     coef = +2.752
    life                      coef = +2.702
    enjoyable                 coef = +2.596
En guclu NEGATIF feature'lar:
    bad                       coef = -5.915
    worst                     coef = -4.349
    boring                    coef = -4.255
    plot                      coef = -4.124
    poor                      coef = -3.737
    looks                     coef = -3.503
    supposed                  coef = -3.475
    stupid                    coef = -3.438
    ridiculous                coef = -3.334
    waste                     coef = -3.249


In [10]:
# --- KARSILASTIRMA: NB vs LogReg vs Optimal (hepsi gercek test setinde) ---
from sklearn.naive_bayes import MultinomialNB

# Ayni TF-IDF temsiliyle Naive Bayes
pipe_nb = Pipeline([
    ('vec', TfidfVectorizer(stop_words= 'english', ngram_range= (1,2),
                 max_features=10_000)),
    ('clf', MultinomialNB())
])
pipe_nb.fit(X_train,y_train)

print(f"TF-IDF + NB:            {pipe_nb.score(X_test,y_test):.2%}")
print(f"TF-IDF + LogReg:        {pipe.score(X_test,y_test):.2%}")
print(f"Optimal LogReg:         {grid.best_estimator_.score(X_test,y_test):.2%}")

TF-IDF + NB:            80.00%
TF-IDF + LogReg:        83.17%
Optimal LogReg:         85.50%
